In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_F.xpt"
csv_file = DATA_DIR / "DEMO_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve an untouched copy of exactly what pandas
# decoded from the original XPT.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 DEMO_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIDEXMON",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDAGEEX",
    "RIDRETH1",
    "DMQMILIT",
    "DMDBORN2",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDSCHOL",
    "DMDMARTL",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "INDHHIN2",
    "INDFMIN2",
    "INDFMPIR",
    "RIDEXPRG",
    "DMDHRGND",
    "DMDHRAGE",
    "DMDHRBR2",
    "DMDHREDU",
    "DMDHRMAR",
    "DMDHSEDU",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANG",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DEMO_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 10537:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 43:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 10,537 rows x 43 columns."
)


# ============================================================
# 4. CHECK FOR KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Do NOT globally replace this value without validating
# the variables and CDC codebook first.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)

true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VALIDATE EXPECTED TINY-ZERO PATTERN
# ============================================================

#
# Validated against the attached DEMO_F XPT
# and CDC 2009-2010 documentation.
#
# Expected tiny values:
#
# RIDAGEYR:
#     Age in years.
#     CDC valid range begins at 0.
#
# RIDAGEMN:
#     Age in months.
#     CDC valid range begins at 0.
#
# RIDAGEEX:
#     Age at exam in months.
#     CDC valid range begins at 0.
#
# DMDEDUC3:
#     Code 0 = Never attended / Kindergarten only.
#
# INDFMPIR:
#     Family income-to-poverty ratio.
#     CDC valid range begins at 0.
#
# WTMEC2YR:
#     MEC sample weight.
#     Interview-only participants have zero MEC weight.
#

EXPECTED_TINY_COUNTS = {
    "RIDAGEYR": 428,
    "RIDAGEMN": 29,
    "RIDAGEEX": 14,
    "DMDEDUC3": 239,
    "INDFMPIR": 93,
    "WTMEC2YR": 284
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "the validated DEMO_F pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


print(
    "\nPASS: Tiny-value pattern matches "
    "validated DEMO_F zero fields."
)


# ============================================================
# 6. RESTORE VALIDATED ZERO VALUES
# ============================================================

#
# Restore ONLY the validated variables above.
#
# No blanket global replacement.
#

print("\n--- ZERO RESTORATION ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    print(
        f"{col}: {actual_count:,} "
        "tiny values restored to 0"
    )


print(
    "\nPASS: All validated zero values restored."
)


# ============================================================
# 7. CREATE EXPECTED CORRECTED XPT COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# data SHOULD contain.
#
# It differs from source_df ONLY where validated
# tiny-zero artifacts were restored to zero.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 8. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE
# ------------------------------------------------------------
#
# NEVER round genuine decimal values.
#
# This is especially important for:
#
#     INDFMPIR
#     WTINT2YR
#     WTMEC2YR
#
# and later Dietary / Laboratory data.
#
# The purpose of this check is to determine which
# variables genuinely contain fractions.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()


    if values.empty:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\n{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )

        print(
            values[
                fractional_mask
            ].head(10)
        )


# ============================================================
# 9. VALIDATE EXPECTED DECIMAL VARIABLES
# ============================================================

#
# These variables genuinely contain decimals in DEMO_F.
#
# DO NOT convert them to Int64.
# DO NOT round them.
#

DECIMAL_COLUMNS = [
    "INDFMPIR",
    "WTINT2YR",
    "WTMEC2YR"
]


if set(
    fractional_columns
) != set(
    DECIMAL_COLUMNS
):

    raise ValueError(
        "STOP: Fractional-variable pattern differs "
        "from expected DEMO_F pattern.\n"
        f"Expected decimal columns: {DECIMAL_COLUMNS}\n"
        f"Actual: {fractional_columns}"
    )


print(
    "\nPASS: Genuine decimal variables identified:"
)

print(
    DECIMAL_COLUMNS
)


print(
    "\nIMPORTANT: These decimal variables will "
    "NOT be rounded or converted to integers."
)


# ============================================================
# 10. CLEAN SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 11. FORMAT WHOLE-NUMBER VARIABLES WITHOUT .0
# ============================================================

#
# IMPORTANT:
#
# Only variables proven to contain whole-number
# values are converted to Int64.
#
# Genuine decimal variables remain float values
# exactly as read from the XPT.
#
# NO .round() is used.
#

INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
    )


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )
    ):

        raise ValueError(
            f"{col} contains a genuine decimal "
            "and cannot safely be converted to Int64."
        )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number demographic variables "
    "formatted without unnecessary .0."
)


print(
    "PASS: Genuine decimal variables were "
    "left unchanged."
)


# ============================================================
# 12. CDC CHECK - SDDSRVYR
# ============================================================

#
# CDC:
#
# 6 = NHANES 2009-2010
# Count = 10,537
#

actual = (
    df["SDDSRVYR"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != {
    6: 10537
}:

    raise ValueError(
        "SDDSRVYR differs from CDC."
    )


print(
    "\nPASS: SDDSRVYR matches CDC."
)


# ============================================================
# 13. CDC CHECK - RIDSTATR
# ============================================================

#
# CDC:
#
# 1 = Interviewed only              284
# 2 = Interviewed + MEC examined 10,253
#

EXPECTED_RIDSTATR = {
    1: 284,
    2: 10253
}


actual = (
    df["RIDSTATR"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_RIDSTATR:

    raise ValueError(
        "RIDSTATR differs from CDC."
    )


print(
    "PASS: RIDSTATR matches CDC."
)


# ============================================================
# 14. CDC CHECK - RIDEXMON
# ============================================================

#
# CDC:
#
# 1 = Nov-Apr : 4,967
# 2 = May-Oct : 5,286
# Missing     :   284
#

EXPECTED_RIDEXMON = {
    1: 4967,
    2: 5286
}


actual = (
    df["RIDEXMON"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_RIDEXMON
    or
    int(
        df["RIDEXMON"]
        .isna()
        .sum()
    )
    != 284
):

    raise ValueError(
        "RIDEXMON differs from CDC."
    )


print(
    "PASS: RIDEXMON matches CDC."
)


# ============================================================
# 15. CDC CHECK - RIAGENDR
# ============================================================

#
# CDC:
#
# Male   = 5,225
# Female = 5,312
#

EXPECTED_RIAGENDR = {
    1: 5225,
    2: 5312
}


actual = (
    df["RIAGENDR"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_RIAGENDR:

    raise ValueError(
        "RIAGENDR differs from CDC."
    )


print(
    "PASS: RIAGENDR matches CDC."
)


# ============================================================
# 16. CDC CHECK - RIDAGEYR
# ============================================================

#
# CDC:
#
# 0-79 = 10,112
# 80   =    425
# Missing = 0
#
# The 428 tiny artifacts represented age = 0.
#

s = df["RIDAGEYR"]


print("\n--- RIDAGEYR ---")


print(
    "Age 0:",
    int(
        s.eq(0).sum()
    )
)


print(
    "Age 0-79:",
    int(
        s.between(
            0,
            79
        ).sum()
    )
)


print(
    "Age 80:",
    int(
        s.eq(80).sum()
    )
)


if not (
    int(
        s.eq(0).sum()
    )
    == 428
    and
    int(
        s.between(
            0,
            79
        ).sum()
    )
    == 10112
    and
    int(
        s.eq(80).sum()
    )
    == 425
    and
    int(
        s.isna().sum()
    )
    == 0
):

    raise ValueError(
        "RIDAGEYR differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: RIDAGEYR matches CDC."
)


# ============================================================
# 17. CDC CHECK - RIDAGEMN
# ============================================================

#
# CDC:
#
# 0-959 = 10,112
# Missing = 425
#

s = df["RIDAGEMN"]


if not (
    int(
        s.eq(0).sum()
    )
    == 29
    and
    int(
        s.between(
            0,
            959
        ).sum()
    )
    == 10112
    and
    int(
        s.isna().sum()
    )
    == 425
):

    raise ValueError(
        "RIDAGEMN differs from CDC."
    )


print(
    "PASS: RIDAGEMN matches CDC."
)


# ============================================================
# 18. CDC CHECK - RIDAGEEX
# ============================================================

#
# CDC:
#
# 0-959 = 9,855
# Missing = 682
#

s = df["RIDAGEEX"]


if not (
    int(
        s.eq(0).sum()
    )
    == 14
    and
    int(
        s.between(
            0,
            959
        ).sum()
    )
    == 9855
    and
    int(
        s.isna().sum()
    )
    == 682
):

    raise ValueError(
        "RIDAGEEX differs from CDC."
    )


print(
    "PASS: RIDAGEEX matches CDC."
)


# ============================================================
# 19. CDC CHECK - RIDRETH1
# ============================================================

EXPECTED_RIDRETH1 = {
    1: 2384,
    2: 1133,
    3: 4420,
    4: 1957,
    5: 643
}


actual = (
    df["RIDRETH1"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_RIDRETH1:

    raise ValueError(
        "RIDRETH1 differs from CDC."
    )


print(
    "PASS: RIDRETH1 matches CDC."
)


# ============================================================
# 20. CDC CHECK - DMDEDUC3
# ============================================================

#
# CDC explicitly reports:
#
# 0 = Never attended / Kindergarten only
# Count = 239
#
# This confirms the 239 tiny values were zeros.
#

EXPECTED_DMDEDUC3 = {
    0: 239,
    1: 216,
    2: 211,
    3: 222,
    4: 213,
    5: 193,
    6: 165,
    7: 185,
    8: 166,
    9: 155,
    10: 193,
    11: 185,
    12: 36,
    13: 94,
    14: 13,
    15: 95,
    55: 2,
    66: 11,
    99: 1
}


actual = (
    df["DMDEDUC3"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_DMDEDUC3
    or
    int(
        df["DMDEDUC3"]
        .isna()
        .sum()
    )
    != 7942
):

    raise ValueError(
        "DMDEDUC3 differs from CDC."
    )


print(
    "PASS: DMDEDUC3 matches CDC."
)


# ============================================================
# 21. CDC CHECK - HOUSEHOLD / FAMILY SIZE
# ============================================================

s = df["DMDHHSIZ"]


if not (
    int(
        s.between(
            1,
            6
        ).sum()
    )
    == 9458
    and
    int(
        s.eq(7).sum()
    )
    == 1079
    and
    int(
        s.isna().sum()
    )
    == 0
):

    raise ValueError(
        "DMDHHSIZ differs from CDC."
    )


print(
    "PASS: DMDHHSIZ matches CDC."
)


s = df["DMDFMSIZ"]


if not (
    int(
        s.between(
            1,
            6
        ).sum()
    )
    == 9561
    and
    int(
        s.eq(7).sum()
    )
    == 976
    and
    int(
        s.isna().sum()
    )
    == 0
):

    raise ValueError(
        "DMDFMSIZ differs from CDC."
    )


print(
    "PASS: DMDFMSIZ matches CDC."
)


# ============================================================
# 22. CDC CHECK - INDFMPIR
# ============================================================

#
# IMPORTANT:
#
# INDFMPIR is a TRUE DECIMAL VARIABLE.
#
# Do NOT round it.
# Do NOT convert it to Int64.
#
# CDC:
#
# 0-4.99 = 8,230
# 5.00+  = 1,311
# Missing = 996
#
# The 93 tiny values are legitimate 0.00 values.
#

s = df["INDFMPIR"]


print("\n--- INDFMPIR ---")


print(
    "Zero:",
    int(
        s.eq(0).sum()
    )
)


print(
    "0-4.99:",
    int(
        s.between(
            0,
            4.99
        ).sum()
    )
)


print(
    "5:",
    int(
        s.eq(5).sum()
    )
)


print(
    "Missing:",
    int(
        s.isna().sum()
    )
)


if not (
    int(
        s.eq(0).sum()
    )
    == 93
    and
    int(
        s.between(
            0,
            4.99
        ).sum()
    )
    == 8230
    and
    int(
        s.eq(5).sum()
    )
    == 1311
    and
    int(
        s.isna().sum()
    )
    == 996
):

    raise ValueError(
        "INDFMPIR differs from CDC."
    )


print(
    "PASS: INDFMPIR matches CDC "
    "and genuine decimals remain intact."
)


# ============================================================
# 23. CDC CHECK - WTINT2YR
# ============================================================

#
# WTINT2YR is a genuine decimal sample-weight variable.
#
# Never round it.
#

s = df["WTINT2YR"]


if (
    int(
        s.isna().sum()
    )
    != 0
    or
    int(
        s.le(0).sum()
    )
    != 0
):

    raise ValueError(
        "WTINT2YR validation failed."
    )


print(
    "\nWTINT2YR minimum:",
    s.min()
)

print(
    "WTINT2YR maximum:",
    s.max()
)


print(
    "PASS: WTINT2YR preserved as decimal."
)


# ============================================================
# 24. CDC CHECK - WTMEC2YR
# ============================================================

#
# WTMEC2YR is also a genuine decimal variable.
#
# The 284 interview-only participants have zero
# MEC sample weight.
#
# CDC RIDSTATR:
#
# Interview-only count = 284
#
# Therefore:
#
# WTMEC2YR zero count should also = 284.
#

s = df["WTMEC2YR"]


print("\n--- WTMEC2YR ---")


print(
    "Zero:",
    int(
        s.eq(0).sum()
    )
)


print(
    "Non-zero:",
    int(
        s.gt(0).sum()
    )
)


print(
    "Missing:",
    int(
        s.isna().sum()
    )
)


if not (
    int(
        s.eq(0).sum()
    )
    == 284
    and
    int(
        s.gt(0).sum()
    )
    == 10253
    and
    int(
        s.isna().sum()
    )
    == 0
):

    raise ValueError(
        "WTMEC2YR validation failed."
    )


#
# Strong consistency check:
#
# All zero WTMEC2YR records should correspond
# to participants who were interviewed only.
#

zero_weight_rows = (
    df.loc[
        df["WTMEC2YR"].eq(0),
        "RIDSTATR"
    ]
)


if not (
    len(
        zero_weight_rows
    )
    == 284
    and
    zero_weight_rows
    .eq(1)
    .all()
):

    raise ValueError(
        "WTMEC2YR zero records do not correspond "
        "exactly to interview-only participants."
    )


print(
    "PASS: WTMEC2YR zero restoration "
    "matches interview-only participants."
)


print(
    "PASS: Non-zero WTMEC2YR decimals "
    "remain unchanged."
)


# ============================================================
# 25. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected changes from the raw pandas XPT:
#
# RIDAGEYR : tiny -> 0
# RIDAGEMN : tiny -> 0
# RIDAGEEX : tiny -> 0
# DMDEDUC3 : tiny -> 0
# INDFMPIR : tiny -> 0
# WTMEC2YR : tiny -> 0
#
# NOTHING ELSE may change numerically.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if col in DECIMAL_COLUMNS:

        current_values = (
            df[col]
            .to_numpy(
                dtype=float
            )
        )

    else:

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 43 variables preserve expected "
    "corrected XPT values exactly."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Whole-number variables are Int64.
#
# Genuine decimals:
#
#     INDFMPIR
#     WTINT2YR
#     WTMEC2YR
#
# remain decimal.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 27. RE-READ EXPORTED CSV
# ============================================================

#
# float_precision="round_trip" ensures pandas
# reads decimal CSV values with round-trip precision
# for fidelity comparison.
#

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 28. EXACT CORRECTED XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 43 CSV columns match "
    "the expected corrected XPT values."
)


# ============================================================
# 29. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 30. CHECK CSV FOR TINY ARTIFACT
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- CSV TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 31. WHOLE-NUMBER CSV FORMAT CHECK
# ============================================================

#
# IMPORTANT:
#
# We cannot simply search the whole file for ".0"
# because genuine decimal variables are allowed to
# contain values such as:
#
#     5.0
#     0.0
#
# Instead, inspect ONLY the columns that are supposed
# to be whole-number values.
#

print("\n--- WHOLE-NUMBER FORMAT CHECK ---")


dot_zero_errors = []


with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(
        f
    )


    for row_number, row in enumerate(
        reader,
        start=2
    ):

        for col in INTEGER_COLUMNS:

            value = row[col]


            if (
                value != ""
                and
                value.endswith(".0")
            ):

                dot_zero_errors.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


                if len(
                    dot_zero_errors
                ) >= 20:

                    break


        if len(
            dot_zero_errors
        ) >= 20:

            break


print(
    "Whole-number '.0' formatting errors:",
    len(
        dot_zero_errors
    )
)


if dot_zero_errors:

    print(
        dot_zero_errors
    )

    raise ValueError(
        "Unexpected .0 formatting remains "
        "in whole-number columns."
    )


print(
    "PASS: SEQN and all whole-number fields "
    "have no unnecessary .0."
)


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

#
# Decimal values are allowed.
#
# Scientific notation is not expected for this
# DEMO_F CSV.
#

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. DECIMAL VALUE FIDELITY CHECK
# ============================================================

#
# Explicitly verify that ALL genuine decimal columns
# survived XPT -> CSV exactly.
#

print("\n--- DECIMAL FIDELITY CHECK ---")


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col} decimal values changed "
            "during CSV export."
        )


    print(
        f"PASS: {col} decimal values preserved exactly."
    )


# ============================================================
# 34. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 35. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)

print(
    "DEMO_F CONVERSION COMPLETE"
)

print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny values restored:",
    sum(
        EXPECTED_TINY_COUNTS.values()
    )
)


print(
    "Decimal variables preserved:",
    ", ".join(
        DECIMAL_COLUMNS
    )
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: 0 expected"
)


print(
    "Genuine decimal values changed: 0 expected"
)

NHANES 2009-2010 DEMO_F XPT -> CSV
Rows: 10,537
Columns: 43
PASS: Structure = 10,537 rows x 43 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 1,087

Tiny values by variable:
RIDAGEYR    428
WTMEC2YR    284
DMDEDUC3    239
INDFMPIR     93
RIDAGEMN     29
RIDAGEEX     14

PASS: Tiny-value pattern matches validated DEMO_F zero fields.

--- ZERO RESTORATION ---
RIDAGEYR: 428 tiny values restored to 0
RIDAGEMN: 29 tiny values restored to 0
RIDAGEEX: 14 tiny values restored to 0
DMDEDUC3: 239 tiny values restored to 0
INDFMPIR: 93 tiny values restored to 0
WTMEC2YR: 284 tiny values restored to 0

PASS: All validated zero values restored.

--- FRACTIONAL VALUE CHECK ---

INDFMPIR: 8,073 genuine fractional values
0    1.36
1    1.07
2    2.27
3    0.81
4    0.69
5    1.01
6    1.91
7    1.36
8    2.68
9    1.27
Name: INDFMPIR, dtype: float64

WTINT2YR: 10,537 genuine fractional values
0    80100.543512
1    53901.104285
2    13953.078343
3    11664.8